# 01 — Phase 1 Data Audit

## tl;dr

The files are structurally usable: 6,602,582 movement rows parse cleanly, candidate keys are unique, and all 490 movement UPCs join to unique metadata. The gate is **PASS WITH CONDITIONS** because 141,285 rows are flagged `OK=0`, 1,851,380 rows have non-positive `PRICE`, `SALE` is blank in 94.55% of rows and includes undocumented `G`/`L` codes, the panel is unbalanced, and the current CSV row count differs from the official manual. No cleaning or substantive price analysis is performed here.

## Context & Methods

This diagnostic notebook is a bounded presentation layer over the reusable streaming utility `../src/audit_data.py`. It reads machine-readable summaries from `../data/processed/audit/`; the utility processes the 6.6M-row movement file in chunks and never modifies `data/raw/`.

Official source: [Dominick's Data Manual and Codebook](https://www.chicagobooth.edu/-/media/enterprise/centers/kilts/datasets/dominicks-dataset/dominicks-manual-and-codebook_kiltscenter).

### Key Assumptions

- Empty CSV fields are counted as structurally missing; no domain meaning is assigned automatically.
- Empty `SALE` is **not** interpreted as no promotion.
- `OK=0` is counted and profiled but not removed.
- Decimal and hexadecimal fields are compared but neither representation is selected here.
- Robust fences identify review candidates, not invalid observations or deletion rules.

## Data

To regenerate every audit output from the repository root:

```bash
python3 src/audit_data.py --chunk-size 250000
```

Set `RUN_FULL_AUDIT=True` below to run the same command inside the notebook. The default is false so routine notebook rendering does not rescan 458 MB unnecessarily.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
AUDIT = ROOT / 'data' / 'processed' / 'audit'
RUN_FULL_AUDIT = False

if RUN_FULL_AUDIT:
    subprocess.run([sys.executable, str(ROOT / 'src' / 'audit_data.py'), '--root', str(ROOT)], check=True)

required = ['audit_manifest.json', 'schema_profile.csv', 'missingness_summary.csv',
            'duplicate_summary.csv', 'join_integrity.csv', 'value_validity.csv',
            'panel_summary.csv', 'hex_validation.csv', 'candidate_exclusion_summary.csv']
assert all((AUDIT / name).exists() for name in required), 'Regenerate the audit outputs first.'

In [ ]:
def read_summary(name, **kwargs):
    return pd.read_csv(AUDIT / name, **kwargs)

integrity = read_summary('file_integrity.csv')
schema = read_summary('schema_profile.csv')
missingness = read_summary('missingness_summary.csv')
duplicates = read_summary('duplicate_summary.csv')
joins = read_summary('join_integrity.csv')
validity = read_summary('value_validity.csv')
panel = read_summary('panel_summary.csv')
hex_validation = read_summary('hex_validation.csv')
candidate_exclusions = read_summary('candidate_exclusion_summary.csv')
codes = read_summary('code_distribution_by_dimension.csv', keep_default_na=False)
metadata = read_summary('upc_metadata_summary.csv')

## Results

### 1. Integrity, schema, and missingness

In [ ]:
display(integrity)
display(schema[['file', 'column', 'documented_type', 'parse_problem_count']])
display(missingness[missingness.missing_count > 0])

### 2. Keys, joins, and metadata

In [ ]:
display(duplicates)
display(joins)
display(metadata)

### 3. Validity flags and code frequencies

Counts are audit flags only; no row is removed.

In [ ]:
display(validity)
display(codes[codes.dimension.eq('ALL')])
display(candidate_exclusions)

### 4. Panel and duplicated numeric fields

In [ ]:
display(panel)
display(hex_validation)

## Takeaways

- Structural ingestion, candidate keys, metadata uniqueness, and UPC joins pass.
- Officially flagged rows, zero-price states, undocumented promotion codes, panel gaps, and a source-manual row-count discrepancy require explicit Phase 2 policy.
- The decimal/hex differences are extremely small, but the primary field choice remains open.
- See `../docs/data-audit.md` for evidence, severity, the gate decision, and proposed—not applied—cleaning policy.
- **Scope:** this notebook documents the data audit and cleaning-policy recommendations; price-index, rigidity, promotion, and econometric analyses are documented separately.